# Ridge Regression — Hands-on Programming

**Goal.** Build a `RidgeRegression` estimator from scratch with three things working correctly:

1. The **centring/standardising trick** so the intercept is *not* penalised (the standard practical convention from `02_mathematics.ipynb` §0).
2. A **Cholesky-based path solver** that reuses $X^T X$ across all candidate $\lambda$ values (`03_optimization.ipynb` §4).
3. **Closed-form LOO-CV (PRESS)** for choosing $\lambda$ in one pass (`04_statistics.ipynb` §4.2).

Then cross-check against `sklearn.linear_model.Ridge` and `sklearn.linear_model.RidgeCV` on synthetic and real data (the diabetes benchmark), and verify the Hoerl–Kennard theorem numerically: tuned ridge has lower test MSE than OLS.

**Role of this notebook.** Implementation and empirical validation. Every formula used here was derived in a previous notebook:

| Used here | Where derived |
|---|---|
| $\hat{\theta}_{ridge}$ = ($X^T X$ + n $\lambda$ $I_p$)^{-1} $X^T y$ | `02_mathematics.ipynb` (2.2) |
| Cholesky path solver | `03_optimization.ipynb` §4 |
| PRESS = (1/n) $\sum$ ($\hat{r}_{i}$ / (1 - $H_{ii}$))^2 | `04_statistics.ipynb` (4.1) |
| Hoerl–Kennard: some $\lambda$ > 0 beats OLS in MSE | `04_statistics.ipynb` Theorem 3.2 |

**Prerequisites.** All four prior notebooks in this folder, plus `01_linear_regression/05_hands_on_programming.ipynb` (the `LinearRegressionOLS` baseline).

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → **`05_hands_on_programming`**.

**Plan.**

1. Imports, seeding.
2. `RidgeRegression` class — Cholesky fit, centred features, PRESS LOO-CV.
3. Sanity-check against `sklearn.linear_model.Ridge` on synthetic data.
4. LOO-CV path on synthetic data — does PRESS pick a sensible $\lambda$?
5. Diabetes: ridge with `RidgeCV`-selected $\lambda$ vs. OLS; verify Hoerl–Kennard on a real benchmark.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve

from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Implementation — `RidgeRegression`

Three implementation notes that match the textbook conventions:

**(a) Don't penalise the intercept.** If we ran the closed form (2.2) of `02_mathematics.ipynb` *with* a column of ones for the bias, the penalty $\lambda$ $\|\theta\|^2$ would shrink the intercept toward zero — which means shifting y by a constant changes the fit. Wrong. Standard fix: centre y to mean zero, centre the columns of X to mean zero, fit ridge *without* a bias column, then back out the intercept as ȳ. (sklearn's `Ridge` does this when `fit_intercept=True`.)

**(b) sklearn uses $\alpha$, we use $\lambda$.** sklearn's `Ridge($\alpha$=$\alpha$)` minimises `$\|X\theta - y\|^2$ + $\alpha$ $\|\theta\|^2$` (no 1/n on the data term). Our formulation uses `(1/n) $\cdot$ $\|X\theta - y\|^2$ + $\lambda$ $\cdot$ $\|\theta\|^2$`. So $\alpha$ = n $\cdot$ $\lambda$. We expose `lam` (our convention) and translate to sklearn's `$\alpha$` only at the cross-check.

**(c) Cholesky is the right backend.** $X^T X$ + n $\lambda$ $I_p$ is symmetric positive-definite (Theorem 2.2 of `02_mathematics.ipynb`). `scipy.linalg.cho_factor` + `cho_solve` is roughly 2$\times$ faster than `np.linalg.solve` (which falls back to LU) and numerically more stable.

In [ ]:
class RidgeRegression:
    """Ridge regression with the centring trick, Cholesky solver, and PRESS LOO-CV.

    Convention:  L(θ) = (1/n) ‖X θ − y‖² + lam · ‖θ‖²
    Intercept is fit unpenalised via centring of X and y.
    """

    def __init__(self, lam=1.0):
        self.lam = lam

    def fit(self, X, y):
        self.x_mean_ = X.mean(axis=0)
        self.y_mean_ = float(y.mean())
        Xc = X - self.x_mean_
        yc = y - self.y_mean_
        n, p = Xc.shape
        A = Xc.T @ Xc + n * self.lam * np.eye(p)        # PD (Theorem 2.2 of 02_math)
        self._chol_ = cho_factor(A)
        self.coef_ = cho_solve(self._chol_, Xc.T @ yc)
        self.intercept_ = self.y_mean_ - self.x_mean_ @ self.coef_
        self._Xc_, self._yc_ = Xc, yc
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_

    def hat_diag(self):
        """Diagonal of H_λ = Xc (XcᵀXc + nλI)⁻¹ Xcᵀ, never forming the n × n matrix."""
        Xc = self._Xc_
        AinvXt = cho_solve(self._chol_, Xc.T)              # (p, n)
        return np.einsum("ij,ji->i", Xc, AinvXt)           # (n,)

    def loo_press(self):
        """PRESS = (1/n) Σ (r̂ᵢ / (1 − Hᵢᵢ))² — eq. (4.1) of 04_statistics."""
        r = self._yc_ - self._Xc_ @ self.coef_
        return float(np.mean((r / (1 - self.hat_diag())) ** 2))

    def eff_dof(self):
        """Effective degrees of freedom df(λ) = trace(H_λ). Eq. (4.1) of 02_math."""
        return float(self.hat_diag().sum())


# Smoke test on tiny synthetic data
n, p = 50, 5
X = rng.normal(size=(n, p))
y = X @ np.arange(1, p + 1) + 3.0 + rng.normal(0, 0.2, size=n)
rr = RidgeRegression(lam=0.05).fit(X, y)
print(f"intercept = {rr.intercept_:.4f}  (true = 3.0)")
print(f"coef      = {rr.coef_}")
print(f"true coef = {np.arange(1, p + 1)}")
print(f"df(λ)     = {rr.eff_dof():.2f}")
print(f"PRESS     = {rr.loo_press():.4f}")

## 2. Sanity check against `sklearn.linear_model.Ridge`

Same data, multiple $\lambda$ values. Translate $\lambda$ → $\alpha$ via $\alpha$ = n $\cdot$ $\lambda$ (see implementation note (b) above). Predictions should agree to machine precision.

In [ ]:
for lam in [0.001, 0.05, 1.0, 100.0]:
    ours = RidgeRegression(lam=lam).fit(X, y)
    skl  = Ridge(alpha=n * lam).fit(X, y)
    pred_ours = ours.predict(X)
    pred_skl  = skl.predict(X)
    print(f"λ = {lam:>7g}  →  max |pred ours − pred sklearn| = {np.max(np.abs(pred_ours - pred_skl)):.2e}"
          f"   intercept diff = {abs(ours.intercept_ - skl.intercept_):.2e}")

## 3. PRESS picks a sensible $\lambda$ on synthetic data

Sweep $\lambda$ on a synthetic problem where we *know* the true MSE-minimising $\lambda$ from Monte Carlo (`04_statistics.ipynb` §3.4). Plot PRESS vs. true MSE.

In [ ]:
# Synthetic regression with controlled true theta and sigma.
n, p = 200, 20
X = rng.normal(size=(n, p))
theta = rng.normal(size=p)
sigma = 1.0
lams = np.logspace(-4, 2, 40)

# Single dataset, PRESS path.
y = X @ theta + rng.normal(0, sigma, size=n)
press_curve = []
df_curve    = []
for lam in lams:
    rr = RidgeRegression(lam=lam).fit(X, y)
    press_curve.append(rr.loo_press())
    df_curve.append(rr.eff_dof())
press_curve = np.array(press_curve)

# Monte Carlo oracle MSE(θ̂ - θ) using a fresh y on each repeat.
M = 200
mse_mc = np.zeros(len(lams))
for m in range(M):
    y_m = X @ theta + rng.normal(0, sigma, size=n)
    for j, lam in enumerate(lams):
        rr = RidgeRegression(lam=lam).fit(X, y_m)
        mse_mc[j] += np.sum((rr.coef_ - theta) ** 2)
mse_mc /= M

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].loglog(lams, press_curve, "o-", color="crimson", label="PRESS (single dataset)")
axes[0].loglog(lams, mse_mc,       "o-", color="steelblue", label="MC oracle MSE(θ̂ − θ)")
axes[0].axvline(lams[np.argmin(press_curve)], color="crimson", ls=":", label=f"PRESS picks λ ≈ {lams[np.argmin(press_curve)]:.2g}")
axes[0].axvline(lams[np.argmin(mse_mc)],     color="steelblue", ls=":", label=f"MC   picks λ ≈ {lams[np.argmin(mse_mc)]:.2g}")
axes[0].set_xlabel("λ (log)"); axes[0].set_ylabel("error (log)")
axes[0].set_title("PRESS ≈ Monte Carlo MSE on this problem")
axes[0].legend(fontsize=8)

axes[1].semilogx(lams, df_curve, color="seagreen")
axes[1].axhline(p, color="black", ls=":")
axes[1].set_xlabel("λ (log)"); axes[1].set_ylabel("df(λ)")
axes[1].set_title(f"Effective dof drops from {p} toward 0")
plt.tight_layout()
plt.show()

**Reading.** PRESS (one dataset) and the Monte Carlo oracle (a wished-for quantity that needs M training sets) pick essentially the same $\lambda$. The two curves are not identical — PRESS is one *random* realisation of an unbiased estimator of MSE, the MC curve is its mean — but the argmin's coincide. The right-hand plot shows df($\lambda$) sliding from p = 20 (at $\lambda$ → 0, OLS) toward 0 (at $\lambda$ → $\infty$).

## 4. Diabetes — verify Hoerl–Kennard on a real benchmark

Fit OLS, ridge with `RidgeCV`-selected $\lambda$ (sklearn's analogue of our PRESS path), and our own implementation with PRESS-selected $\lambda$. Compare RMSE and $R^2$ on a held-out test set. We expect tuned ridge to beat OLS — that is Theorem 3.2 of `04_statistics.ipynb`.

In [ ]:
data = load_diabetes()
X_full, y_full = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=SEED)
n_tr = X_tr.shape[0]

# Candidate λ on the log-scale.
lams = np.logspace(-5, 2, 50)

# Ours: pick λ that minimises PRESS on the training set.
press_path = [RidgeRegression(lam=l).fit(X_tr, y_tr).loo_press() for l in lams]
lam_ours   = lams[int(np.argmin(press_path))]
ours = RidgeRegression(lam=lam_ours).fit(X_tr, y_tr)

# sklearn: RidgeCV (uses LOO CV by default with cv=None).
skl_cv = RidgeCV(alphas=n_tr * lams, cv=None).fit(X_tr, y_tr)
lam_skl = skl_cv.alpha_ / n_tr

# OLS baseline.
ols = LinearRegression().fit(X_tr, y_tr)

def metrics(est, name):
    y_pred_tr = est.predict(X_tr); y_pred_te = est.predict(X_te)
    return {
        "name":    name,
        "rmse_tr": float(np.sqrt(mean_squared_error(y_tr, y_pred_tr))),
        "rmse_te": float(np.sqrt(mean_squared_error(y_te, y_pred_te))),
        "r2_te":   float(r2_score(y_te, y_pred_te)),
    }

rows = [
    metrics(ols,    "OLS"),
    metrics(ours,   f"Ours Ridge (PRESS-λ = {lam_ours:.4g})"),
    metrics(skl_cv, f"sklearn RidgeCV (λ = {lam_skl:.4g})"),
]

print(f"{'method':<40}  {'RMSE_train':>11}  {'RMSE_test':>11}  {'R2_test':>8}")
for r in rows:
    print(f"{r['name']:<40}  {r['rmse_tr']:>11.2f}  {r['rmse_te']:>11.2f}  {r['r2_te']:>8.4f}")

In [ ]:
# Visualise the PRESS path and the chosen λ
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogx(lams, press_path, color="crimson")
ax.axvline(lam_ours, color="crimson", ls=":", label=f"ours PRESS-best λ = {lam_ours:.4g}")
ax.axvline(lam_skl, color="steelblue", ls=":", label=f"sklearn RidgeCV λ = {lam_skl:.4g}")
ax.set_xlabel("λ (log)")
ax.set_ylabel("PRESS LOO-MSE")
ax.set_title("Diabetes — PRESS path and the chosen λ")
ax.legend(fontsize=8)
plt.show()

**Reading.** Several things to confirm in the table:

1. **Our `RidgeRegression(lam=lam_ours)` and `RidgeCV` pick nearly the same $\lambda$.** Both use LOO CV; differences are floating-point and the grid resolution.
2. **Our predictions and sklearn's match closely** — different $\lambda$ grids may give slightly different test RMSE, but the OLS baseline is the same.
3. **The tuned ridge model improves on OLS** — usually by a small but real amount on diabetes. This is the Hoerl–Kennard theorem (Theorem 3.2 of `04_statistics.ipynb`) on real data.

The improvement on diabetes is modest because the data is well-behaved (n ≫ p, no severe multicollinearity). On a higher-dimensional or more collinear problem, the OLS → ridge gain is much larger; this is exactly why ridge / glmnet became the default linear regressor in modern statistical software.

## 5. The coefficient path on diabetes

All ten diabetes coefficients as $\lambda$ sweeps. The picture says: every coefficient shrinks continuously toward zero, none is exactly killed (contrast with the Lasso, next folder).

In [ ]:
lams_path = np.logspace(-5, 3, 80)
coef_path = np.array([RidgeRegression(lam=l).fit(X_tr, y_tr).coef_ for l in lams_path])

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for j, name in enumerate(data.feature_names):
    ax.plot(lams_path, coef_path[:, j], lw=1, label=name)
ax.axvline(lam_ours, color="black", ls=":", label=f"PRESS-best λ = {lam_ours:.4g}")
ax.set_xscale("log")
ax.set_xlabel("λ (log)")
ax.set_ylabel("coefficient value")
ax.set_title("Ridge regularisation path on diabetes")
ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.axhline(0, color="black", lw=0.5)
plt.show()

## Takeaway

- **Implementation.**   Centre y and X, fit with Cholesky on (XcᵀXc + n $\lambda$ $I_p$), back out the intercept as ȳ - x̄ᵀ $\hat{\theta}$. $\approx$ 30 lines of code.
- **PRESS is free LOO-CV.**   Once we have the Cholesky factor, the diagonal of $H_{\lambda}$ is one back-solve; LOO MSE follows from eq. (4.1) of `04_statistics.ipynb`. No n-fold loop needed.
- **Cross-check.**   Predictions match `sklearn.linear_model.Ridge($\alpha$ = n $\cdot$ $\lambda$)` to machine precision; PRESS-best $\lambda$ matches `RidgeCV`'s LOO-selected $\lambda$.
- **Hoerl–Kennard in practice.**   On diabetes, tuned ridge has lower test RMSE than OLS — small but real gain. On higher-collinearity datasets the gain is dramatic.
- **Coefficient path.**   Every coefficient slides smoothly toward zero as $\lambda$ increases. None is ever *exactly* zero. That is the defining contrast with the Lasso (next folder).

**This concludes the Ridge Regression folder.** Next: `04_lasso_regression/` replaces $\lambda$ $\|\theta\|^2$ with $\lambda$ $\|\theta\|_1$ — the constraint region (`02_mathematics.ipynb` §5) becomes a polytope with vertices on the axes, so the fitted $\hat{\theta}$ tends to have *exact* zeros: variable selection and shrinkage in one optimisation.